# 🎵 MusicLens: Exploratory Data Analysis & Statistical Diagnostics

This notebook provides a complete exploratory data analysis of the **Spotify 30,000 Songs Dataset** (~32.8k tracks across 6 genres and 24 subgenres).

### Core Analytical Questions Addressed:
1. **Volume**: Total songs (raw vs deduplicated).
2. **Artists**: Total artists and catalog distribution.
3. **Genres**: Macro-genres and subgenre taxonomy.
4. **Genre Sizes**: Which genres dominate track volume?
5. **Genre Popularity**: Which genres have the highest average popularity? (Hypothesis tests included)
6. **Top Artists**: Which artists achieve highest popularity? (Sample-size controlled)
7. **Distributions**: Statistical shape (mean, median, IQR, skewness, kurtosis) of all 10 audio features.
8. **Correlations**: What audio features drive track popularity? (Pearson & Spearman).
9. **Cross-Genre Variations**: How do acoustic profiles vary by genre? (ANOVA & Radar Profiles).
10. **Outlier Detection**: Tukey's fences & acoustic edge-cases (skits, live shows, spoken word).

In [ ]:
import sys
from pathlib import Path

# Set project root path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from pipeline.config import CLEANED_DATA_DIR, AUDIO_FEATURE_COLS
from pipeline.utils.stats import compute_distribution_metrics, compute_correlations_with_target, test_genre_differences

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

## 1. Data Ingestion & Overview

In [ ]:
# Load cleaned dataset and unique tracks dimension
df = pd.read_csv(CLEANED_DATA_DIR / "spotify_songs_cleaned.csv")
tracks = pd.read_csv(CLEANED_DATA_DIR / "tracks.csv")
audio = pd.read_csv(CLEANED_DATA_DIR / "audio_features.csv")

print(f"Total Playlist-Track Records: {len(df):,}")
print(f"Unique Songs (track_id):      {len(tracks):,}")
print(f"Unique Artists:               {tracks['track_artist'].nunique():,}")
print(f"Unique Genres:                {df['playlist_genre'].nunique()}")
print(f"Unique Subgenres:             {df['playlist_subgenre'].nunique()}")
tracks.head(3)

## 2. Genre Distribution & Track Volume
Analysis of representation across macro-genres.

In [ ]:
genre_summary = df.groupby("playlist_genre").agg(
    total_entries=("track_id", "count"),
    unique_tracks=("track_id", "nunique"),
    avg_popularity=("track_popularity", "mean"),
    median_popularity=("track_popularity", "median")
).reset_index().sort_values(by="total_entries", ascending=False)

genre_summary["pct_share"] = (genre_summary["total_entries"] / len(df) * 100).round(2)
genre_summary["avg_popularity"] = genre_summary["avg_popularity"].round(2)
display(genre_summary)

## 3. Genre Popularity Comparison & Hypothesis Testing
Testing if the difference in popularity across genres is statistically significant using One-Way ANOVA and Kruskal-Wallis tests.

In [ ]:
genre_groups = [df[df["playlist_genre"] == g]["track_popularity"].values for g in df["playlist_genre"].unique()]
f_stat, anova_p = stats.f_oneway(*genre_groups)
h_stat, kw_p = stats.kruskal(*genre_groups)

print(f"One-Way ANOVA F-Statistic: {f_stat:.2f} (p-value: {anova_p:.2e})")
print(f"Kruskal-Wallis H-Statistic: {h_stat:.2f} (p-value: {kw_p:.2e})")
print("Conclusion: Significant difference in popularity across genres (p < 0.001).")

## 4. Artist Popularity & Catalog Depth
Evaluating artist performance with a threshold filter (minimum 5 tracks) to eliminate single-song viral anomalies.

In [ ]:
artist_perf = tracks.groupby("track_artist")["track_popularity"].agg(track_count="count", avg_pop="mean", median_pop="median").reset_index()
top_filtered_artists = artist_perf[artist_perf["track_count"] >= 5].sort_values(by="avg_pop", ascending=False).head(15)
top_filtered_artists["avg_pop"] = top_filtered_artists["avg_pop"].round(2)
display(top_filtered_artists)

## 5. Audio Feature Distributions & Diagnostics
Computing parametric and non-parametric distribution statistics (mean, std, median, skewness, kurtosis, and Tukey's fences outliers).

In [ ]:
dist_records = []
for feat in ["track_popularity", "duration_ms"] + AUDIO_FEATURE_COLS:
    m = compute_distribution_metrics(tracks[feat])
    dist_records.append({
        "Feature": feat,
        "Mean": m["mean"],
        "Std": m["std"],
        "Median": m["median"],
        "IQR": m["iqr"],
        "Skewness": m["skewness"],
        "Kurtosis": m["kurtosis"],
        "Skew Diagnosis": m["skew_description"],
        "Outliers (%)": m["outliers_mild_pct"]
    })

dist_df = pd.DataFrame(dist_records)
display(dist_df)

## 6. Audio Features Correlation with Popularity

In [ ]:
corr_df = compute_correlations_with_target(tracks, feature_cols=AUDIO_FEATURE_COLS, target_col="track_popularity")
display(corr_df)

## 7. Cross-Genre Audio Profiles
Comparing average acoustic features by genre.

In [ ]:
genre_audio_means = df.groupby("playlist_genre")[AUDIO_FEATURE_COLS].mean().round(3)
display(genre_audio_means)